In [3]:
import pandas as pd
from datetime import datetime, timezone

data_inputs = "../data/inputs/"
data_onputs = "../data/onputs/psa-tests"

data_santiago = "ventas_santiago.csv"
data_buenos_aires = "ventas_buenos_aires.json"
data_lima = "ventas_lima.parquet"

In [4]:
#Schemas globales
from typing import NamedTuple
class SalesOutput(NamedTuple):
    id_transaccion: str
    ciudad: str
    monto_original: str
    moneda_origen: str
    fecha_transaccion: str
    ingestado_at: str

In [5]:
#Funciones globales
#Funcion para datetime
def ingestion_time():
    #Funcion que devuelve el el momento actual exacto en UTC ISO
    return datetime.now(timezone.utc).isoformat()

# Testeando ingesta de santiago (CSV)

In [6]:
df_santiago_full = pd.read_csv(data_inputs + data_santiago)
df_santiago = df_santiago_full.head(100)
df_santiago.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_transaccion  100 non-null    object
 1   ciudad          100 non-null    object
 2   monto           100 non-null    int64 
 3   fecha           100 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.3+ KB


In [7]:
#Schemas Santiago

class SalesSantiagoInput(NamedTuple):
    id_transaccion: str
    ciudad: str
    monto: str
    fecha: str


In [8]:
#---Funciones especificas para parsear Santiago (CSV)---
def parse_csv_santiago(line):
    row = next(csv.reader([line]))
    return SalesSantiagoInput(
        id_transaccion=row[0],
        ciudad=row[1],
        monto=row[2],
        fecha=row[3],
    )

def santiago_to_output(row):
    return SalesOutput(
        id_transaccion=row.id_transaccion,
        ciudad=row.ciudad,
        monto_original=row.monto,
        moneda_origen="CLP",
        fecha_transaccion=row.fecha,
        ingestado_at=ingestion_time(),
    )

In [9]:
df_output_santiago = pd.DataFrame(
    [
        santiago_to_output(SalesSantiagoInput(*row))
        for row in df_santiago.itertuples(index=False, name=None)
    ]
)

df_output_santiago.head()

,id_transaccion,ciudad,monto_original,moneda_origen,fecha_transaccion,ingestado_at
0,STGO-000001,Santiago,-43481,CLP,2026-05-12,2026-06-28T18:45:36.244424+00:00
1,STGO-000002,Santiago,-12734,CLP,2026-05-27,2026-06-28T18:45:36.244424+00:00
2,STGO-000003,Santiago,88332,CLP,2026-05-11,2026-06-28T18:45:36.244424+00:00
3,STGO-000004,Santiago_Err,100349,CLP,2026-06-10,2026-06-28T18:45:36.244424+00:00
4,STGO-000005,Santiago,-17431,CLP,2026-06-02,2026-06-28T18:45:36.244424+00:00


# Testeando ingesta de Buenos Aires (JSON)

In [10]:
df_buenos_aires_full = pd.read_json(data_inputs + data_buenos_aires, lines=True)
df_buenos_aires = df_buenos_aires_full.head(100)
df_buenos_aires.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   tx_id      100 non-null    object        
 1   sucursal   100 non-null    object        
 2   total      100 non-null    object        
 3   timestamp  100 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 3.3+ KB


In [11]:
#Schemas Buenos Aires
class SalesBuenosAiresInput(NamedTuple):
    tx_id: str
    sucursal: str
    total: str
    timestamp: str


In [12]:
#---Funciones especificas para parsear Buenos Aires (json)---
def parse_json_buenos_aires(line):
    row = next(json.loads([line]))
    return SalesBuenosAiresInput(
        id_transaccion=row[0],
        ciudad=row[1],
        monto=row[2],
        fecha=row[3],
    )

def buenos_aires_to_output(row):
    return SalesOutput(
        id_transaccion=row.id_transaccion,
        ciudad=row.ciudad,
        monto_original=row.monto,
        moneda_origen="ARS",
        fecha_transaccion=row.fecha,
        ingestado_at=ingestion_time(),
    )

In [13]:
df_output_buenos_aires = pd.DataFrame(
    [
        buenos_aires_to_output(SalesSantiagoInput(*row))
        for row in df_buenos_aires.itertuples(index=False, name=None)
    ]
)

df_output_buenos_aires.head()

,id_transaccion,ciudad,monto_original,moneda_origen,fecha_transaccion,ingestado_at
0,BA-000001,Buenos Aires,82775,ARS,2026-06-08,2026-06-28T18:45:36.300555+00:00
1,BA-000002,Buenos Aires,11902,ARS,2026-05-22,2026-06-28T18:45:36.300555+00:00
2,BA-000003,Buenos Aires,49615,ARS,2026-05-21,2026-06-28T18:45:36.300555+00:00
3,BA-000004,Buenos Aires,19245,ARS,2026-06-09,2026-06-28T18:45:36.300555+00:00
4,BA-000005,Buenos Aires,13094,ARS,2026-05-19,2026-06-28T18:45:36.300555+00:00


# Testeando ingesta de Lima (Parque)

In [14]:
df_lima_full = pd.read_parquet(data_inputs + data_lima, engine="pyarrow")
df_lima = df_lima_full.head(100)
df_lima.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  100 non-null    object 
 1   city            100 non-null    object 
 2   local_value     100 non-null    float64
 3   sales_date      100 non-null    object 
dtypes: float64(1), object(3)
memory usage: 3.3+ KB


In [15]:
#----Lima----
class SalesLimaInput(NamedTuple):
    transaccion_id: str
    city: str
    local_value: str
    sales_date: str


In [16]:
def lima_to_output(row):
    return SalesOutput(
        id_transaccion=row["id_transaccion"],
        ciudad=row["ciudad"],
        monto_original=row["monto"],
        moneda_origen="PEN",
        fecha_transaccion=row["fecha"],
        ingestado_at=ingestion_time(),
    )